# Chapter 7 - Lab 4: <font color='blue'>Evaluator-Optimizer Extension</font>

**<font color='purple'>Goal</font>**:
Take the sequential pipeline from Lab 1 and **bolt on an evaluator** — a fifth agent that reviews the Supervisor's output and routes back for revision if quality criteria are not met. This transforms the architecture from pure sequential into a hybrid: *sequential + evaluator-optimizer* (see the chapter's *Choosing and combining architectural styles* section).

You will see how a simple deterministic check inside a tool, combined with a single extra agent, dramatically improves output quality.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex** — `FunctionAgent`, `AgentWorkflow`, `Context`.
* **OpenAI** `gpt-4o`.
* **Stub data** — same as Lab 1.

Prerequisite: read Lab 1 first — the four pipeline agents are reused here.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 7 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%207/common.py',
        'common.py',
    )

from common import (
    PROFITABILITY_THRESHOLDS, LIQUIDITY_THRESHOLDS,
    format_threshold_prompt, stub_lookup,
)

print('Common helpers loaded.')

## 4. Re-create the four pipeline agents from Lab 1

Same agent code as Lab 1 — abbreviated here. The full versions of these helpers live in Lab 1 if you want the comments. We define them again so this lab is self-contained.

In [ ]:
from llama_index.core.workflow import Context
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

async def get_fundamentals(ctx: Context, ticker: str) -> str:
    data = stub_lookup(ticker)
    state = await ctx.store.get('state')
    state['ticker'] = ticker
    state['ratios'] = {**data['profitability'], **data['liquidity']}
    await ctx.store.set('state', state)
    return 'fundamentals ok'

async def get_profitability_ratios(ctx: Context) -> str:
    state = await ctx.store.get('state')
    state['profitability_ratios'] = {k: state['ratios'][k] for k in PROFITABILITY_THRESHOLDS}
    state['threshold_profitability_comments'] = format_threshold_prompt(PROFITABILITY_THRESHOLDS, 'Profitability')
    await ctx.store.set('state', state)
    return 'profitability ok'

async def get_liquidity_ratios(ctx: Context) -> str:
    state = await ctx.store.get('state')
    state['liquidity_ratios'] = {k: state['ratios'][k] for k in LIQUIDITY_THRESHOLDS}
    state['threshold_liquidity_comments'] = format_threshold_prompt(LIQUIDITY_THRESHOLDS, 'Liquidity')
    await ctx.store.set('state', state)
    return 'liquidity ok'

async def get_overall_comments(ctx: Context) -> str:
    state = await ctx.store.get('state')
    state['overall_comments'] = 'synthesis complete'
    await ctx.store.set('state', state)
    return 'overall ok'

## 5. The Evaluator's tool — deterministic quality checks

The evaluator does *not* use the LLM to judge quality — it uses **deterministic code** to check the state object. This is cheaper and far more reliable than asking a model 'is this complete?'. The checks: are all 4 profitability metrics scored? all 4 liquidity metrics? did the Supervisor produce overall comments?

In [ ]:
async def evaluate_assessment(ctx: Context) -> str:
    state = await ctx.store.get('state')
    issues = []

    if state.get('threshold_profitability_comments') is None:
        issues.append('Missing profitability analysis.')
    if state.get('threshold_liquidity_comments') is None:
        issues.append('Missing liquidity analysis.')
    if not state.get('overall_comments'):
        issues.append('Missing overall assessment.')

    if len(state.get('profitability_ratios', {})) < 4:
        issues.append(f"Only {len(state.get('profitability_ratios', {}))}/4 profitability metrics scored.")
    if len(state.get('liquidity_ratios', {})) < 4:
        issues.append(f"Only {len(state.get('liquidity_ratios', {}))}/4 liquidity metrics scored.")

    if issues:
        return 'REVISION NEEDED:\n' + '\n'.join(f'- {i}' for i in issues)
    return 'APPROVED. Assessment complete and consistent.'

## 6. Build the agents (now with EvaluatorAgent)

The four original agents are unchanged except the SupervisorAgent now hands off to `EvaluatorAgent` instead of ending the run. The EvaluatorAgent can hand back to the SupervisorAgent for revision — this is the optimizer side of the pattern. Note the termination discipline: the evaluator's prompt caps the loop at **one revision round** and makes the evaluator write the final answer on approval, so the cycle cannot spin forever.

In [ ]:
fundamental_agent = FunctionAgent(
    name='FundamentalAgent', description='Collect fundamentals.',
    system_prompt=(
        'Call get_fundamentals exactly once for the requested ticker, '
        'then hand off to ProfitabilityAgent.'
    ),
    llm=llm, tools=[get_fundamentals],
    can_handoff_to=['ProfitabilityAgent'],
)
profitability_agent = FunctionAgent(
    name='ProfitabilityAgent', description='Score profitability.',
    system_prompt=(
        'Call get_profitability_ratios exactly once, score profitability, '
        'then hand off to LiquidityAgent.'
    ),
    llm=llm, tools=[get_profitability_ratios],
    can_handoff_to=['LiquidityAgent'],
)
liquidity_agent = FunctionAgent(
    name='LiquidityAgent', description='Score liquidity.',
    system_prompt=(
        'Call get_liquidity_ratios exactly once, score liquidity, '
        'then hand off to SupervisorAgent.'
    ),
    llm=llm, tools=[get_liquidity_ratios],
    can_handoff_to=['SupervisorAgent'],
)
supervisor_agent = FunctionAgent(
    name='SupervisorAgent', description='Synthesize overall verdict.',
    system_prompt=(
        'Call get_overall_comments exactly once, write the combined '
        'assessment, then hand off to EvaluatorAgent for review.'
    ),
    llm=llm, tools=[get_overall_comments],
    can_handoff_to=['EvaluatorAgent'],
)
evaluator_agent = FunctionAgent(
    name='EvaluatorAgent',
    description='Review assessment quality.',
    system_prompt=(
        'You are a quality reviewer. Call evaluate_assessment exactly once. '
        'If it returns APPROVED, you are the last agent: write the final '
        'approved assessment yourself and do not hand off. If it returns '
        'REVISION NEEDED, hand off to SupervisorAgent with the specific '
        'issues — but allow at most one revision round before approving.'
    ),
    llm=llm, tools=[evaluate_assessment],
    can_handoff_to=['SupervisorAgent'],
)

## 7. Run the extended workflow

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult

workflow = AgentWorkflow(
    agents=[fundamental_agent, profitability_agent, liquidity_agent,
            supervisor_agent, evaluator_agent],
    root_agent='FundamentalAgent',
    initial_state={
        'ratios': {}, 'ticker': '',
        'profitability_ratios': {}, 'liquidity_ratios': {},
        'threshold_profitability_comments': None,
        'threshold_liquidity_comments': None,
        'overall_comments': None,
    },
)

# Five agents plus a possible revision round need headroom over the
# default budget of 20 workflow iterations.
handler = workflow.run(user_msg='Analyze AAPL financial health.',
                       max_iterations=60)

# Print every tool call as it completes — you will see the four pipeline
# tools, the handoffs, and finally evaluate_assessment's verdict.
async for event in handler.stream_events():
    if isinstance(event, ToolCallResult):
        print(f'[{event.tool_name}] -> {str(event.tool_output)[:90]}')
result = await handler
print('\n\nFinal:', result)

## 8. Results

You added one agent and one deterministic tool, and the pipeline gained a self-correction loop. **What to notice about the evaluator-optimizer pattern:**

* **Cheap deterministic checks beat LLM-as-judge** for binary completeness questions.
* **The pattern is incremental.** You did not redesign the pipeline — you bolted on a quality   gate. Real production systems grow this way: start sequential, add an evaluator when you see   failure modes.
* **Bounded loops only.** This lab caps the cycle at one revision round through the   evaluator's prompt; production systems enforce the cap in code and pair it with an explicit   iteration budget (`max_iterations`). Without a cap, a stubborn supervisor could loop forever.
* **The next lab** parallelizes *across companies* — orthogonal to the evaluator pattern,   which can stack on top.